In [ ]:
!python3 -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in
%pip install python-woc pandas matplotlib tqdm

In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    "mobvoi_wenet",
    "pbs-assess_gfsynopsis",
    "pmlmodelling_ersem",
    "vearch_vearch",
    "hannorein_rebound",
    "soedinglab_mmseqs2",
    "hpddm_hpddm",
    "stephenslab_mr.mash.alpha",
    "gem_oq-engine",
    "sixin-zh_kymatio_wph",
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks

# chunks = [df['sha1'][x:x+40] for x in range(0, len(df), 40)]
commit_data = []

done = {item["commit"] for item in commit_data}
remaining = df.loc[~df["sha1"].isin(done), "sha1"]

chunks = [
    remaining.iloc[x:x+40]
    for x in range(0, len(remaining), 40)
]

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(4)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(
    df[["sha1", "project"]], left_on="commit", right_on="sha1"
)
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

In [ ]:
import pandas as pd
from pathlib import Path

print("Current folder:", Path.cwd())

for filename in [
    "df_commit_data.csv",
    "bjy819_project_summary.csv",
    "df_commits.csv"
]:
    path = Path(filename)
    print(filename, "— exists" if path.exists() else "— not found")

retrieved = pd.read_csv("df_commit_data.csv")

print("Saved commits:", len(retrieved))
print("Columns:", retrieved.columns.tolist())

In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
import time

missing_ids = sorted({sha for project, sha in missing})
recovered = []

for start in range(0, len(missing_ids), 10):
    batch = missing_ids[start:start + 10]
    res, err = woc.get_values_many("commit.tch", batch)

    for sha, values in res.items():
        if not values:
            continue

        commit = values[0]
        recovered.append({
            "commit": sha,
            "author": commit[2][0],
            "author_time": int(commit[2][1]),
            "message": commit[4],
        })

    print(f"Batch {start // 10 + 1}: returned {len(res)}/{len(batch)}")
    if err:
        print("Errors:", err)

    time.sleep(4)

# Save recovered records separately before merging
pd.DataFrame(recovered).to_csv(
    "recovered_commits.csv", index=False
)

print("Recovered:", len(recovered), "of", len(missing_ids))

In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

## GitHub project information

Current observations; each row records its collection time in UTC. Last commit uses the default branch's latest listed commit and its committer timestamp. This is not the historical September 30, 2025 status assessment.

| Project | Stars | Forks | Last commit (UTC) | Branch | Repository | Status |
|---|---:|---:|---|---|---|---|
| mobvoi_wenet | 5243 | 1190 | 2026-09-07T07:21:08Z | main | https://github.com/wenet-e2e/wenet | ok |
| pbs-assess_gfsynopsis | 17 | 1 | 2026-09-19T23:19:56Z | master | https://github.com/pbs-assess/gfsynopsis | ok |
| pmlmodelling_ersem | 30 | 14 | 2026-08-18T13:51:19Z | master | https://github.com/pmlmodelling/ersem | ok |
| vearch_vearch | 2331 | 365 | 2026-07-27T05:29:53Z | master | https://github.com/vearch/vearch | ok |
| hannorein_rebound | 1138 | 270 | 2026-10-06T12:13:52Z | main | https://github.com/hannorein/rebound | ok |
| soedinglab_mmseqs2 | 2148 | 303 | 2026-09-30T04:36:10Z | master | https://github.com/soedinglab/MMseqs2 | ok |
| hpddm_hpddm | 160 | 37 | 2026-09-26T19:46:30Z | main | https://github.com/hpddm/hpddm | ok |
| stephenslab_mr.mash.alpha | 7 | 5 | 2026-02-16T16:00:01Z | master | https://github.com/stephenslab/mr.mashr | ok |
| gem_oq-engine | 451 | 325 | 2026-10-08T02:09:51Z | master | https://github.com/gem/oq-engine | ok |
| sixin-zh_kymatio_wph | 0 | 0 | 2025-04-10T14:37:11Z | master | https://github.com/sixin-zh/kymatio_wph | ok |
